In [1]:
%pip install anthropic python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-4-5-20251001"

In [14]:
from datetime import datetime
from anthropic.types import ToolParam

def get_current_datetime(date_format = "%d-%m-%Y %H:%M:%S"):
    if not date_format:
        raise ValueError("date_format should be valid")
    return datetime.now().strftime(date_format)


get_current_datetime_schema = ToolParam({
  "name": "get_current_datetime",
  "description": "Returns the current date and time as a formatted string. If no format is provided, defaults to 'DD-MM-YYYY HH:MM:SS' (e.g., '05-10-2026 14:32:07'). Use this tool whenever the user asks for the current date, time, or both, or needs a timestamp in a specific format. The format must be a valid, non-empty strftime-style format string; an empty or missing format will result in an error.",
  "input_schema": {
    "type": "object",
    "properties": {
      "date_format": {
        "type": "string",
        "description": "A Python strftime-style format string specifying how the date/time should be formatted. Examples: '%d-%m-%Y %H:%M:%S' (e.g., 05-10-2026 14:32:07), '%Y-%m-%d' (e.g., 2026-10-05), '%I:%M %p' (e.g., 02:32 PM), '%A, %B %d, %Y' (e.g., Monday, October 05, 2026). Defaults to '%d-%m-%Y %H:%M:%S' if not specified. Must not be an empty string.",
        "default": "%d-%m-%Y %H:%M:%S"
      }
    },
    "required": []
  }
})

In [13]:
get_current_datetime()

'05-10-2026 18:14:07'

In [29]:
messages = []
add_messages(messages, "Give me the current time in HH:MM format")

response = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    tools=[get_current_datetime_schema]
    )
response

Message(id='msg_011CfjLka247vsCedjPhiPYk', container=None, content=[ToolUseBlock(id='toolu_016Jpj9DNn4MtkhrGVbXtwQD', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M'}, name='get_current_datetime', type='tool_use', toolset_name=None)], model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason='tool_use', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=836, output_tokens=61, output_tokens_details=None, server_tool_use=None, service_tier='standard'), diagnostics=None)

In [30]:
messages.append({"role": "assistant", "content": response.content})

In [31]:
messages

[{'role': 'user', 'content': 'Give me the current time in HH:MM format'},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_016Jpj9DNn4MtkhrGVbXtwQD', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M'}, name='get_current_datetime', type='tool_use', toolset_name=None)]}]

In [32]:
messages.append(
    {"role": "user",
     "content": [
         {
             "type": "tool_result",
             "tool_use_id": response.content[0].id,
             "content": get_current_datetime(**response.content[0].input),
             "is_error": False
         }
     ]
    }
)

messages

[{'role': 'user', 'content': 'Give me the current time in HH:MM format'},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_016Jpj9DNn4MtkhrGVbXtwQD', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M'}, name='get_current_datetime', type='tool_use', toolset_name=None)]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_016Jpj9DNn4MtkhrGVbXtwQD',
    'content': '21:10',
    'is_error': False}]}]

In [33]:
result = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    tools=[get_current_datetime_schema]
    )
result

Message(id='msg_011CfjLpci97G7DAwbj212ks', container=None, content=[TextBlock(citations=None, text='The current time is **21:10**.', type='text')], model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=912, output_tokens=12, output_tokens_details=None, server_tool_use=None, service_tier='standard'), diagnostics=None)

In [36]:
result.content[0].text

'The current time is **21:10**.'

In [39]:
from anthropic.types import Message

def add_messages(messages, message):
    message = {"role": "user", "content": message.content if isinstance(message, Message) else message}
    messages.append(message)

def add_assistant_messages(messages, message):
    message = {"role": "assistant", "content": message.content if isinstance(message, Message) else message}
    messages.append(message)

def chat(messages, system = "", temperature = 0, stop_sequences = [], tools=[]):
    message = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    system=system,
    temperature=temperature,
    stop_sequences=stop_sequences,
    tools=tools
    )
    return message

def text_from_message(message):
    return "\n".join([
        block.text for block in message if block.type == 'text'
    ])

In [58]:
import json

tools_dict = {"get_current_datetime": get_current_datetime}

def run_tool(tool_name, tool_input):
    tool_result = tools_dict[tool_name](**tool_input)
    return tool_result


def run_tools(messages):
    tool_use_blocks = [block for block in messages if block.type == 'tool_use']
    tool_result_blocks = []
    for tool_use_block in tool_use_blocks:
        try:
            tool_output = run_tool(tool_use_block.name, tool_use_block.input)
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_use_block.id,
                "content": json.dumps(tool_output),
                "is_error": False
            }
        except Exception as e:
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_use_block.id,
                "content": f"Error: {e}",
                "is_error": True
            }
        tool_result_blocks.append(tool_result_block)
    return tool_result_blocks
    

In [61]:
def run_conversation(messages):
    while True:
        response = chat(messages, tools=[get_current_datetime_schema])
        add_assistant_messages(messages, response)
        print(text_from_message(response.content))

        if response.stop_reason != "tool_use":
            break
        tool_result_blocks = run_tools(response.content)
        add_messages(messages, tool_result_blocks)
    return messages

In [62]:
messages = []

add_messages(messages, "What is the current time in HH:MM format? Also give the current time in SS format.")

run_conversation(messages)


The current time is:
- **HH:MM format**: 20:28
- **SS format**: 17

So the current time is 20:28:17 (8:28:17 PM).


[{'role': 'user',
  'content': 'What is the current time in HH:MM format? Also give the current time in SS format.'},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_01Uiqs145eLMFkHopGnfHuQZ', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M'}, name='get_current_datetime', type='tool_use', toolset_name=None),
   ToolUseBlock(id='toolu_01KpC8S6eyRTMWECRRwHRvqx', caller=DirectCaller(type='direct'), input={'date_format': '%S'}, name='get_current_datetime', type='tool_use', toolset_name=None)]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_01Uiqs145eLMFkHopGnfHuQZ',
    'content': '"20:28"',
    'is_error': False},
   {'type': 'tool_result',
    'tool_use_id': 'toolu_01KpC8S6eyRTMWECRRwHRvqx',
    'content': '"17"',
    'is_error': False}]},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text='The current time is:\n- **HH:MM format**: 20:28\n- **SS format**: 17\n\nSo the current time is 20:28:17 (8:28:17 PM).'

### Fine-grained tool streaming (`eager_input_streaming`)

#### What it is
A tool-level setting that streams a tool call's arguments **without buffering or JSON validation**. Chunks of large parameters start arriving much sooner. The docs' example shows a ~15s delay without it vs ~3s with it.

#### How to enable
Set `eager_input_streaming: True` on each user-defined tool where you want it, and make the request with streaming on.

```python
tools = [{
    "name": "make_file",
    "description": "Write text to a file",
    "eager_input_streaming": True,
    "input_schema": {
        "type": "object",
        "properties": {
            "filename": {"type": "string"},
            "lines_of_text": {"type": "array", "items": {"type": "string"}},
        },
        "required": ["filename", "lines_of_text"],
    },
}]
```

#### Stream events for a tool call
| Event | Meaning |
|---|---|
| `content_block_start` | `tool_use` block opens (name, id, empty input) |
| `content_block_delta` (`input_json_delta`) | Fragment of the argument JSON (`partial_json`) |
| `content_block_stop` | Block complete, input is final |

#### When to use it
- Showing tool arguments live in a UI (file contents, code, long text)
- Large string/array parameters where default buffering causes long pauses
- Piping arguments downstream as they arrive

#### When not to use it
- If you only run the tool after the block completes, the default is simpler and safer. Time to a runnable call is about the same.
- It does not make generation faster. It only reduces the wait before the first chunks of a large argument arrive.

#### Caveats
- **Input may be partial or invalid JSON.** Validate before executing.
- If the input is invalid, return a `tool_result` with `is_error: true` so the model can retry.
- Without manual handling, `stream.get_final_message()` gives you the parsed `block.input`.

#### Rule of thumb
Need to show or process arguments **as they arrive** → turn it on and validate.
Only need the **final** arguments → leave it off.

### Built-in tools: Text Editor and Web Search

Anthropic provides some tools with a fixed schema (the `type` field selects the tool). The key difference is **who runs the tool**.

| | Text editor tool | Web search tool |
|---|---|---|
| Execution | **Client-side**: your code runs it | **Server-side**: Anthropic runs it |
| You implement | The file operations | Nothing |
| Admin enablement | Not needed | Must be enabled in the Claude Console (privacy settings) |
| Extra cost | Tokens only | Per-search fee on top of tokens (failed searches aren't billed) |

---

#### Text editor tool

Lets Claude view and modify text files (debugging, fixing, improving code or other text documents).

- **Current version:** `text_editor_20250728`
- **Tool name:** `str_replace_based_edit_tool`
- **Optional:** `max_characters` limits how much of a file is returned when viewing

```python
tools = [{
    "type": "text_editor_20250728",
    "name": "str_replace_based_edit_tool",
    "max_characters": 10000,
}]
```

**Commands Claude can emit:** `view`, `str_replace`, `create`, `insert`

**Flow:**
1. Claude returns a `tool_use` block with a command and its arguments.
2. Your code performs the file operation.
3. You send the outcome back as a `tool_result` (use `is_error: True` on failure).

---

#### Web search tool

Gives Claude access to real-time web content, so it can answer with information beyond its knowledge cutoff.

- **Basic version:** `web_search_20250305`
- **Newer versions:** `web_search_20260209` adds dynamic filtering on supported models. One docs snapshot also lists `web_search_20260318`. Check the current docs for the latest.
- **Tool name:** `web_search`
- **Common parameter:** `max_uses` caps searches per request. Domain allow/block lists and `allowed_callers` are also supported.

```python
tools = [{"type": "web_search_20250305", "name": "web_search", "max_uses": 5}]
```

**Notes:**
- Claude decides when to search. The API runs the searches and gives it the results.
- Errors (e.g. rate limits) still return HTTP 200, so check the result block for an error.
- Web search is token-intensive.
- Works with prompt caching and the Messages Batches API.
- **Platform support:** Claude API, Claude Platform on AWS, and Microsoft Foundry. Vertex AI supports only the basic version. Amazon Bedrock doesn't support it.

---

#### Quick rule of thumb
- Need Claude to **read or edit files** in your environment → text editor (you implement the handlers).
- Need **current information from the web** → web search (just enable it).

> Model support and version strings change over time. Verify against the tool reference in the Claude docs before relying on them.